# CLAHE — Equalização de Histograma Adaptativa Limitada por Contraste

Este notebook aplica **CLAHE** em imagens de obras para realçar trincas, fissuras e detalhes sutis em concreto.

O processamento é otimizado para o **Google Colab** e salva automaticamente:

- Imagem em escala de cinza;
- Imagem processada com CLAHE;
- Comparativo visual;
- Comparação lado a lado;
- Relatório em CSV;
- Resumo em TXT.

**Autora:** Adriana Rolim, coordenação  
**Versão:** 2.0 — Otimizado para Colab + Processamento em Lote  
**Data:** Agosto de 2025

## 1. Importação das bibliotecas

Execute esta célula para carregar as bibliotecas necessárias.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from google.colab import drive
import pandas as pd

## 2. Montagem do Google Drive e configuração das pastas

Antes de executar, confira se a estrutura no seu Google Drive está assim:

```text
MyDrive/
└── Python_VC/
    ├── fotos_obra/
    └── output/
```

As imagens originais devem estar dentro da pasta `fotos_obra`.

In [ ]:
# Montar o Google Drive
drive.mount('/content/drive')

# Configurações principais
PASTA_BASE = '/content/drive/MyDrive/Python_VC'
PASTA_ENTRADA = os.path.join(PASTA_BASE, "fotos_obra")
PASTA_SAIDA = os.path.join(PASTA_BASE, "output")

# Parâmetros CLAHE
PARAMS_CLAHE = {
    'clip_limit': 3.0,        # Limite de contraste: 2.0 a 4.0 costuma funcionar bem para trincas
    'tile_grid_size': (8, 8), # Grade 8x8 favorece detalhes finos
    'redimensionar_max': 1200 # Largura máxima para redimensionamento; use 0 para manter original
}

print("Configuração carregada.")
print(f"Pasta base: {PASTA_BASE}")
print(f"Pasta de entrada: {PASTA_ENTRADA}")
print(f"Pasta de saída: {PASTA_SAIDA}")
print(f"Parâmetros CLAHE: {PARAMS_CLAHE}")

## 3. Verificação da configuração

Esta célula verifica se as pastas existem e se há imagens na pasta de entrada.

In [ ]:
def verificar_configuracao():
    """Verifica se todos os arquivos e pastas necessários existem."""
    print("🔍 VERIFICANDO CONFIGURAÇÃO:")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📁 Pasta entrada: {PASTA_ENTRADA}")
    print(f"📁 Pasta saída: {PASTA_SAIDA}")
    print(f"🎛️  Parâmetros CLAHE: {PARAMS_CLAHE}")

    problemas = []

    if not os.path.exists(PASTA_BASE):
        problemas.append("❌ Pasta base não encontrada!")

    if not os.path.exists(PASTA_ENTRADA):
        problemas.append("❌ Pasta de entrada 'fotos_obra' não encontrada!")

    # Verificar se há imagens na pasta de entrada
    if os.path.exists(PASTA_ENTRADA):
        arquivos_imagem = [
            f for f in os.listdir(PASTA_ENTRADA)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))
        ]

        if not arquivos_imagem:
            problemas.append("❌ Nenhuma imagem encontrada na pasta 'fotos_obra'")
        else:
            print(f"📸 Imagens encontradas: {len(arquivos_imagem)}")

    if problemas:
        print("\n".join(problemas))
        return False

    return True


verificar_configuracao()

## 4. Funções auxiliares

Estas funções criam pastas, redimensionam imagens e calculam a entropia visual.

In [ ]:
def garantir_pasta(pasta):
    """Cria a pasta se ela ainda não existir."""
    os.makedirs(pasta, exist_ok=True)


def redimensionar_imagem(imagem, max_width):
    """
    Redimensiona a imagem mantendo a proporção.

    Parâmetros:
        imagem: Imagem OpenCV.
        max_width: Largura máxima. Use 0 para manter o tamanho original.

    Retorno:
        np.array: Imagem redimensionada.
    """
    if max_width and max_width > 0 and imagem.shape[1] > max_width:
        proporcao = max_width / imagem.shape[1]
        nova_largura = max_width
        nova_altura = int(imagem.shape[0] * proporcao)

        imagem_redimensionada = cv2.resize(
            imagem,
            (nova_largura, nova_altura),
            interpolation=cv2.INTER_AREA
        )

        print(
            f"   📐 Redimensionada: "
            f"{imagem.shape[1]}x{imagem.shape[0]} → {nova_largura}x{nova_altura}"
        )

        return imagem_redimensionada

    return imagem


def calcular_entropia(imagem):
    """
    Calcula a entropia da imagem como medida de informação visual.

    Parâmetros:
        imagem: Imagem em escala de cinza.

    Retorno:
        float: Valor da entropia.
    """
    histograma = cv2.calcHist([imagem], [0], None, [256], [0, 256])
    histograma = histograma[histograma > 0] / np.sum(histograma)

    entropia = -np.sum(histograma * np.log2(histograma))

    return entropia

## 5. Aplicação do CLAHE em uma imagem

A função abaixo aplica o CLAHE em uma imagem individual e calcula métricas antes e depois do processamento.

In [ ]:
def aplicar_clahe_imagem(imagem, nome_arquivo):
    """
    Aplica CLAHE em uma imagem individual.

    Parâmetros:
        imagem: Imagem OpenCV em BGR ou em escala de cinza.
        nome_arquivo: Nome do arquivo para exibição no log.

    Retorno:
        tuple: imagem_original, imagem_clahe, metricas.
    """
    print(f"   🎯 Aplicando CLAHE: {nome_arquivo}")

    # Redimensionar se necessário
    imagem_redimensionada = redimensionar_imagem(
        imagem,
        PARAMS_CLAHE['redimensionar_max']
    )

    # Converter para escala de cinza se necessário
    if len(imagem_redimensionada.shape) == 3:
        imagem_cinza = cv2.cvtColor(imagem_redimensionada, cv2.COLOR_BGR2GRAY)
    else:
        imagem_cinza = imagem_redimensionada.copy()

    contraste_antes = np.std(imagem_cinza)
    entropia_antes = calcular_entropia(imagem_cinza)

    # Calcular métricas antes do CLAHE
    metricas_antes = {
        'contraste': contraste_antes,
        'brilho_medio': np.mean(imagem_cinza),
        'entropia': entropia_antes,
        'dimensoes': f"{imagem_cinza.shape[1]}x{imagem_cinza.shape[0]}"
    }

    # Aplicar CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=PARAMS_CLAHE['clip_limit'],
        tileGridSize=PARAMS_CLAHE['tile_grid_size']
    )

    imagem_clahe = clahe.apply(imagem_cinza)

    contraste_depois = np.std(imagem_clahe)
    entropia_depois = calcular_entropia(imagem_clahe)

    # Evitar divisão por zero
    melhoria_contraste = (
        ((contraste_depois - contraste_antes) / contraste_antes) * 100
        if contraste_antes != 0 else 0
    )

    melhoria_entropia = (
        ((entropia_depois - entropia_antes) / entropia_antes) * 100
        if entropia_antes != 0 else 0
    )

    # Calcular métricas após CLAHE
    metricas_depois = {
        'contraste': contraste_depois,
        'brilho_medio': np.mean(imagem_clahe),
        'entropia': entropia_depois,
        'melhoria_contraste': melhoria_contraste,
        'melhoria_entropia': melhoria_entropia
    }

    print(f"   📊 Contraste: {metricas_antes['contraste']:.2f} → {metricas_depois['contraste']:.2f}")
    print(f"   📈 Melhoria de contraste: {metricas_depois['melhoria_contraste']:+.1f}%")
    print(f"   🧠 Melhoria de entropia: {metricas_depois['melhoria_entropia']:+.1f}%")

    return imagem_cinza, imagem_clahe, {
        'antes': metricas_antes,
        'depois': metricas_depois
    }

## 6. Salvamento dos resultados

Esta função salva as imagens processadas, os comparativos visuais e a comparação simples lado a lado.

In [ ]:
def salvar_resultados_clahe(imagem_original, imagem_clahe, metricas, nome_arquivo, pasta_saida):
    """
    Salva todos os resultados do processamento CLAHE.

    Parâmetros:
        imagem_original: Imagem original em escala de cinza.
        imagem_clahe: Imagem após CLAHE.
        metricas: Dicionário com métricas calculadas.
        nome_arquivo: Nome do arquivo original.
        pasta_saida: Pasta para salvar os resultados.

    Retorno:
        dict: Caminhos dos arquivos gerados.
    """
    nome_base = os.path.splitext(nome_arquivo)[0]

    # Salvar imagem CLAHE
    caminho_clahe = os.path.join(pasta_saida, f"{nome_base}_clahe.png")
    cv2.imwrite(caminho_clahe, imagem_clahe)

    # Salvar imagem original em cinza
    caminho_original = os.path.join(pasta_saida, f"{nome_base}_cinza.png")
    cv2.imwrite(caminho_original, imagem_original)

    # Criar visualização comparativa
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Linha 1: imagens
    axes[0, 0].imshow(imagem_original, cmap='gray')
    axes[0, 0].set_title("Imagem Original em Cinza")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(imagem_clahe, cmap='gray')
    axes[0, 1].set_title("Após CLAHE")
    axes[0, 1].axis("off")

    # Linha 2: histogramas
    axes[1, 0].hist(imagem_original.ravel(), 256, [0, 256], alpha=0.7, label='Original')
    axes[1, 0].set_title("Histograma — Original")
    axes[1, 0].set_xlabel("Intensidade de pixel")
    axes[1, 0].set_ylabel("Frequência")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].hist(imagem_clahe.ravel(), 256, [0, 256], alpha=0.7, label='CLAHE')
    axes[1, 1].set_title("Histograma — Após CLAHE")
    axes[1, 1].set_xlabel("Intensidade de pixel")
    axes[1, 1].set_ylabel("Frequência")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Adicionar métricas
    texto_metricas = f"Clip Limit: {PARAMS_CLAHE['clip_limit']}\n"
    texto_metricas += f"Tile Grid: {PARAMS_CLAHE['tile_grid_size']}\n"
    texto_metricas += f"Dimensões: {metricas['antes']['dimensoes']}\n"
    texto_metricas += f"Melhoria Contraste: {metricas['depois']['melhoria_contraste']:+.1f}%\n"
    texto_metricas += f"Melhoria Entropia: {metricas['depois']['melhoria_entropia']:+.1f}%"

    fig.text(
        0.5,
        0.01,
        texto_metricas,
        ha='center',
        va='bottom',
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.3")
    )

    plt.tight_layout()

    # Salvar figura comparativa
    caminho_comparativo = os.path.join(pasta_saida, f"{nome_base}_comparativo.png")
    plt.savefig(caminho_comparativo, dpi=150, bbox_inches='tight')
    plt.close()

    # Criar versão lado a lado simples
    comparacao_simples = cv2.hconcat([imagem_original, imagem_clahe])
    caminho_comparacao_simples = os.path.join(
        pasta_saida,
        f"{nome_base}_comparacao_simples.jpg"
    )
    cv2.imwrite(caminho_comparacao_simples, comparacao_simples)

    caminhos = {
        'original': caminho_original,
        'clahe': caminho_clahe,
        'comparativo': caminho_comparativo,
        'comparacao_simples': caminho_comparacao_simples
    }

    return caminhos

## 7. Processamento em lote

Esta célula processa todas as imagens da pasta `fotos_obra`.

In [ ]:
def processar_clahe_em_lote():
    """
    Processa todas as imagens em lote aplicando CLAHE.

    Retorno:
        list: Resultados do processamento.
    """
    print("\n🔄 INICIANDO CLAHE EM LOTE...")
    print("=" * 60)

    # Garantir que a pasta de saída existe
    garantir_pasta(PASTA_SAIDA)

    # Encontrar imagens
    arquivos_imagem = [
        f for f in os.listdir(PASTA_ENTRADA)
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))
    ]

    if not arquivos_imagem:
        print("❌ Nenhuma imagem encontrada para processar.")
        return []

    print(f"📁 Total de imagens para processar: {len(arquivos_imagem)}")

    resultados_gerais = []
    processadas = 0
    erros = 0

    for i, arquivo in enumerate(arquivos_imagem, 1):
        print(f"\n[{i}/{len(arquivos_imagem)}] {arquivo}")

        caminho_imagem = os.path.join(PASTA_ENTRADA, arquivo)

        try:
            # Carregar imagem
            imagem = cv2.imread(caminho_imagem)

            if imagem is None:
                print(f"❌ Não foi possível carregar: {arquivo}")
                erros += 1
                continue

            # Aplicar CLAHE
            imagem_original, imagem_clahe, metricas = aplicar_clahe_imagem(imagem, arquivo)

            # Salvar resultados
            caminhos = salvar_resultados_clahe(
                imagem_original,
                imagem_clahe,
                metricas,
                arquivo,
                PASTA_SAIDA
            )

            # Adicionar aos resultados gerais
            resultados_gerais.append({
                'imagem': arquivo,
                'dimensoes': metricas['antes']['dimensoes'],
                'contraste_antes': metricas['antes']['contraste'],
                'contraste_depois': metricas['depois']['contraste'],
                'entropia_antes': metricas['antes']['entropia'],
                'entropia_depois': metricas['depois']['entropia'],
                'melhoria_contraste': metricas['depois']['melhoria_contraste'],
                'melhoria_entropia': metricas['depois']['melhoria_entropia'],
                'caminhos': caminhos
            })

            processadas += 1
            print(f"   ✅ CLAHE aplicado: {arquivo}")

        except Exception as e:
            print(f"❌ Erro ao processar {arquivo}: {str(e)}")
            erros += 1

    # Resumo final
    print("\n" + "=" * 60)
    print("📊 RESUMO DO CLAHE:")
    print(f"   ✅ Imagens processadas com sucesso: {processadas}")
    print(f"   ❌ Erros de processamento: {erros}")
    print(f"   📁 Pasta de saída: {PASTA_SAIDA}")

    if resultados_gerais:
        melhoria_contraste_media = np.mean(
            [r['melhoria_contraste'] for r in resultados_gerais]
        )
        melhoria_entropia_media = np.mean(
            [r['melhoria_entropia'] for r in resultados_gerais]
        )

        print(f"   📈 Melhoria média no contraste: {melhoria_contraste_media:+.1f}%")
        print(f"   🧠 Melhoria média na entropia: {melhoria_entropia_media:+.1f}%")

    return resultados_gerais

## 8. Geração dos relatórios

Esta célula gera:

- `relatorio_clahe.csv`
- `resumo_clahe.txt`

In [ ]:
def gerar_relatorio_clahe(resultados):
    """Gera relatório detalhado do processamento CLAHE."""
    if not resultados:
        print("ℹ️ Não há resultados para gerar relatório.")
        return

    relatorio_path = os.path.join(PASTA_SAIDA, "relatorio_clahe.csv")

    # Preparar dados para CSV
    dados_csv = []

    for res in resultados:
        dados_csv.append({
            'imagem': res['imagem'],
            'dimensoes': res['dimensoes'],
            'contraste_antes': res['contraste_antes'],
            'contraste_depois': res['contraste_depois'],
            'entropia_antes': res['entropia_antes'],
            'entropia_depois': res['entropia_depois'],
            'melhoria_contraste': res['melhoria_contraste'],
            'melhoria_entropia': res['melhoria_entropia']
        })

    # Criar DataFrame com resultados
    df_resultados = pd.DataFrame(dados_csv)
    df_resultados.to_csv(relatorio_path, index=False)

    print(f"📄 Relatório CLAHE salvo: {relatorio_path}")

    # Relatório resumido
    resumo_path = os.path.join(PASTA_SAIDA, "resumo_clahe.txt")

    with open(resumo_path, 'w', encoding='utf-8') as f:
        f.write("RELATÓRIO DE PROCESSAMENTO CLAHE\n")
        f.write("=" * 50 + "\n")
        f.write(f"Data: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Parâmetros: {PARAMS_CLAHE}\n")
        f.write(f"Total de imagens processadas: {len(resultados)}\n")

        if resultados:
            melhoria_contraste_media = np.mean(
                [r['melhoria_contraste'] for r in resultados]
            )
            melhoria_entropia_media = np.mean(
                [r['melhoria_entropia'] for r in resultados]
            )

            f.write("\nESTATÍSTICAS GERAIS:\n")
            f.write(f"  Melhoria média no contraste: {melhoria_contraste_media:+.1f}%\n")
            f.write(f"  Melhoria média na entropia: {melhoria_entropia_media:+.1f}%\n")

        f.write("\nDETALHES POR IMAGEM:\n")

        for res in resultados:
            f.write(
                f"- {res['imagem']}: "
                f"contraste {res['melhoria_contraste']:+.1f}%, "
                f"entropia {res['melhoria_entropia']:+.1f}%\n"
            )

    print(f"📋 Resumo CLAHE salvo: {resumo_path}")

## 9. Visualização de exemplos

Esta célula exibe alguns comparativos gerados.

In [ ]:
def mostrar_exemplos_resultados(num_exemplos=3):
    """Mostra exemplos dos resultados gerados."""
    arquivos_comparativos = [
        f for f in os.listdir(PASTA_SAIDA)
        if f.endswith('_comparativo.png')
    ]

    if not arquivos_comparativos:
        print("ℹ️ Nenhum arquivo comparativo encontrado para exibir.")
        return

    exemplos = arquivos_comparativos[:num_exemplos]

    print(f"\n🖼️  EXIBINDO {len(exemplos)} EXEMPLOS DE CLAHE:")

    for exemplo in exemplos:
        caminho_exemplo = os.path.join(PASTA_SAIDA, exemplo)

        try:
            img = plt.imread(caminho_exemplo)

            plt.figure(figsize=(15, 10))
            plt.imshow(img)
            plt.title(f"Exemplo CLAHE: {exemplo}")
            plt.axis('off')
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"   ❌ Erro ao exibir {exemplo}: {str(e)}")

## 10. Execução completa do processamento

Execute esta célula para rodar o fluxo completo:

1. Verificar configuração;
2. Aplicar CLAHE em lote;
3. Gerar relatórios;
4. Exibir exemplos.

In [ ]:
def main():
    """Função principal do notebook."""
    print("🚀 INICIANDO CLAHE — REALCE DE TRINCAS E DETALHES")
    print("=" * 60)

    if verificar_configuracao():
        # Processar todas as imagens
        resultados = processar_clahe_em_lote()

        if resultados:
            # Gerar relatórios
            gerar_relatorio_clahe(resultados)

            print("\n🎯 CLAHE CONCLUÍDO!")
            print(f"   📁 Resultados salvos em: {PASTA_SAIDA}")
            print(f"   📊 {len(resultados)} imagens processadas com sucesso")
            print("   🔍 Contraste realçado para melhor detecção de trincas")

            # Mostrar exemplos
            mostrar_exemplos_resultados(2)

            return resultados

        else:
            print("\n❌ Nenhuma imagem foi processada com sucesso.")
            return []

    else:
        print("\n❌ Não foi possível iniciar o processamento.")
        print("\n💡 SOLUÇÃO:")
        print("1. Verifique se a pasta 'Python_VC/fotos_obra' existe no seu Drive")
        print("2. Adicione imagens à pasta 'fotos_obra'")
        print("3. Execute o notebook novamente")
        return []


resultados_clahe = main()

## 11. Consulta rápida dos resultados

Após executar o processamento, use esta célula para visualizar o relatório em formato de tabela.

In [ ]:
if 'resultados_clahe' in globals() and resultados_clahe:
    df_clahe = pd.DataFrame([
        {
            'imagem': r['imagem'],
            'dimensoes': r['dimensoes'],
            'contraste_antes': r['contraste_antes'],
            'contraste_depois': r['contraste_depois'],
            'melhoria_contraste': r['melhoria_contraste'],
            'entropia_antes': r['entropia_antes'],
            'entropia_depois': r['entropia_depois'],
            'melhoria_entropia': r['melhoria_entropia']
        }
        for r in resultados_clahe
    ])

    display(df_clahe)
else:
    print("Execute primeiro a célula de processamento completo.")